# Machine Learning

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


## Bengaluru House Prices

In [ ]:
from google.colab import files
files.upload()


In [ ]:
house = pd.read_csv("bengaluru_house_prices.csv")
house.head()


In [ ]:
house = house.drop_duplicates()

house["bhk"] = house["size"].astype(str).str.extract(r"(\d+)").astype(float)

def convert_sqft(x):
    x = str(x)
    if "-" in x:
        a, b = x.split("-")
        return (float(a) + float(b)) / 2
    try:
        return float(x)
    except ValueError:
        return np.nan

house["total_sqft"] = house["total_sqft"].apply(convert_sqft)

house = house.dropna(subset=["total_sqft", "price"])
house = house[(house["total_sqft"] > 200) & (house["price"] > 0)]


In [ ]:
X = house[[
    "area_type",
    "availability",
    "location",
    "bhk",
    "total_sqft",
    "bath",
    "balcony"
]]

y = house["price"]

cat = ["area_type", "availability", "location"]
num = ["bhk", "total_sqft", "bath", "balcony"]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat),
    ("num", SimpleImputer(strategy="median"), num)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [ ]:
model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    model,
    params,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


In [ ]:
pred = grid.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("R2:", r2_score(y_test, pred))


## Churn Modelling

In [ ]:
files.upload()


In [ ]:
churn = pd.read_csv("Churn_Modelling.csv")
churn.head()


In [ ]:
churn = churn.drop_duplicates()

X = churn.drop(columns=[
    "Exited",
    "RowNumber",
    "CustomerId",
    "Surname"
])

y = churn["Exited"]

cat = ["Geography", "Gender"]
num = [c for c in X.columns if c not in cat]

preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat),
    ("num", SimpleImputer(strategy="median"), num)
])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
model = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    model,
    params,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


In [ ]:
pred = grid.predict(X_test)
prob = grid.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print(classification_report(y_test, pred))
